In [ ]:
repository_filter: list[str] = []
data_file_methods: str = "../samples/method_quality_metrics.csv"
data_file_packages: str = "../samples/package_quality_metrics.csv"
data_file_smells: str = "../samples/code_smells.csv"
data_file_gaps: str = "../samples/test_gaps.csv"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import warnings

warnings.simplefilter("ignore")

methods_df = dt.read_csv(data_file_methods)
methods_df = qu.filter_repos(methods_df, repository_filter)
methods_df = qu.add_repo_short(methods_df)

class_df = dt.read_csv("../samples/class_quality_metrics.csv")
class_df = qu.filter_repos(class_df, repository_filter)
class_df = qu.add_repo_short(class_df)

pkg_df = dt.read_csv(data_file_packages)
pkg_df = qu.filter_repos(pkg_df, repository_filter)

smells_df = dt.read_csv(data_file_smells)
smells_df = qu.filter_repos(smells_df, repository_filter)
smells_df = qu.add_repo_short(smells_df)

gaps_df = dt.read_csv(data_file_gaps)
gaps_df = qu.filter_repos(gaps_df, repository_filter)
gaps_df = qu.add_repo_short(gaps_df)

In [ ]:
has_data = len(methods_df) > 0

if not has_data:
    fig = qu.empty_figure("No data available for the executive dashboard.")
else:
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[
            "Portfolio Health Score",
            "Top 5 Riskiest Repositories",
            "Code Smells by Severity",
            "Test Gap Coverage",
            "Worst Dependency Cycles",
            "Method Complexity Distribution",
        ],
        specs=[
            [{"type": "indicator"}, {"type": "bar"}, {"type": "bar"}],
            [{"type": "indicator"}, {"type": "table"}, {"type": "histogram"}],
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    # --- Panel 1: Portfolio Health Score (gauge) ---
    avg_mi = class_df["maintainabilityIndex"].mean() if len(class_df) > 0 else 0
    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=avg_mi,
            number={"suffix": "/100", "font": {"size": 28}},
            gauge=dict(
                axis=dict(range=[0, 100]),
                bar=dict(color="#4CAF50" if avg_mi >= 65 else "#FF8A65"),
                steps=[
                    {"range": [0, 30], "color": "#FFEBEE"},
                    {"range": [30, 65], "color": "#FFF8E1"},
                    {"range": [65, 100], "color": "#E8F5E9"},
                ],
            ),
        ),
        row=1, col=1,
    )

    # --- Panel 2: Top 5 Riskiest Repos (horizontal bar) ---
    if len(methods_df) > 0:
        methods_df["debtScore"] = qu.compute_debt_score(methods_df)
        repo_risk = (
            methods_df.groupby("repoShort")["debtScore"]
            .mean()
            .nlargest(5)
            .sort_values(ascending=True)
        )
        fig.add_trace(
            go.Bar(
                y=repo_risk.index,
                x=repo_risk.values,
                orientation="h",
                marker_color="#EF5350",
            ),
            row=1, col=2,
        )

    # --- Panel 3: Code Smells by Severity (stacked bar) ---
    if len(smells_df) > 0:
        for severity in qu.SEVERITY_ORDER:
            subset = smells_df[smells_df["severity"] == severity]
            counts = subset.groupby("repoShort").size().nlargest(5)
            fig.add_trace(
                go.Bar(
                    x=counts.index,
                    y=counts.values,
                    name=severity,
                    marker_color=qu.SEVERITY_COLORS.get(severity, "#999"),
                    showlegend=True,
                ),
                row=1, col=3,
            )
    fig.update_layout(barmode="stack")

    # --- Panel 4: Test Gap Coverage (gauge) ---
    if len(gaps_df) > 0 and len(methods_df) > 0:
        total_methods = len(methods_df)
        gap_methods = len(gaps_df)
        coverage_pct = max(0, 100 * (1 - gap_methods / total_methods))
    else:
        coverage_pct = 100 if len(methods_df) > 0 else 0
    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=coverage_pct,
            number={"suffix": "%", "font": {"size": 28}},
            gauge=dict(
                axis=dict(range=[0, 100]),
                bar=dict(
                    color="#4CAF50" if coverage_pct >= 70 else "#FF8A65"
                ),
                steps=[
                    {"range": [0, 40], "color": "#FFEBEE"},
                    {"range": [40, 70], "color": "#FFF8E1"},
                    {"range": [70, 100], "color": "#E8F5E9"},
                ],
            ),
        ),
        row=2, col=1,
    )

    # --- Panel 5: Worst Dependency Cycles (table) ---
    cycle_pkgs = pkg_df[pkg_df["inCycle"].astype(str).str.lower() == "true"]
    if len(cycle_pkgs) > 0:
        cycle_top = cycle_pkgs.nlargest(
            min(8, len(cycle_pkgs)),
            ["afferentCoupling"],
        )
        fig.add_trace(
            go.Table(
                header=dict(
                    values=["Package", "Ca", "Ce", "Cycle Members"],
                    fill_color="#EF5350",
                    font=dict(color="white", size=11),
                    align="left",
                ),
                cells=dict(
                    values=[
                        cycle_top["packageName"].apply(
                            lambda x: x.split(".")[-1]
                        ),
                        cycle_top["afferentCoupling"],
                        cycle_top["efferentCoupling"],
                        cycle_top["cycleMembers"].apply(
                            lambda x: str(x)[:40]
                        ),
                    ],
                    align="left",
                    font=dict(size=10),
                ),
            ),
            row=2, col=2,
        )
    else:
        fig.add_trace(
            go.Table(
                header=dict(values=["Status"]),
                cells=dict(values=[["No dependency cycles found"]]),
            ),
            row=2, col=2,
        )

    # --- Panel 6: Method Complexity Distribution (histogram) ---
    if len(methods_df) > 0:
        fig.add_trace(
            go.Histogram(
                x=methods_df["cyclomaticComplexity"],
                nbinsx=30,
                marker_color="#7986CB",
                name="Cyclomatic",
                showlegend=False,
            ),
            row=2, col=3,
        )

    fig.update_layout(
        title="Code Quality Executive Dashboard",
        height=800,
        width=1200,
        margin=dict(l=40, r=40, t=80, b=40),
        plot_bgcolor="white",
    )
fig.show()